In [19]:
code = '''import voyageai
import faiss
import numpy as np
import pandas as pd
import os
import pickle
from dotenv import load_dotenv

load_dotenv()

voyage_client = voyageai.Client(
    api_key=os.getenv("VOYAGE_API_KEY")
)

def prepare_documents(data_path):
    df = pd.read_csv(data_path)
    documents = []
    for _, row in df.iterrows():
        doc = (
            f"Passenger Info:\\n"
            f"- Gender: {row[\'Gender\']}\\n"
            f"- Age: {row[\'Age\']}\\n"
            f"- Customer Type: {row[\'Customer Type\']}\\n"
            f"- Type of Travel: {row[\'Type of Travel\']}\\n"
            f"- Class: {row[\'Class\']}\\n"
            f"- Flight Distance: {row[\'Flight Distance\']} miles\\n\\n"
            f"Service Ratings:\\n"
            f"- Wifi: {row[\'Inflight wifi service\']}/5\\n"
            f"- Food: {row[\'Food and drink\']}/5\\n"
            f"- Seat Comfort: {row[\'Seat comfort\']}/5\\n"
            f"- Entertainment: {row[\'Inflight entertainment\']}/5\\n"
            f"- On-board Service: {row[\'On-board service\']}/5\\n"
            f"- Leg Room: {row[\'Leg room service\']}/5\\n"
            f"- Baggage: {row[\'Baggage handling\']}/5\\n"
            f"- Checkin: {row[\'Checkin service\']}/5\\n"
            f"- Online Boarding: {row[\'Online boarding\']}/5\\n\\n"
            f"Delays:\\n"
            f"- Departure: {row[\'Departure Delay in Minutes\']} mins\\n"
            f"- Arrival: {row[\'Arrival Delay in Minutes\']} mins\\n\\n"
            f"Outcome:\\n"
            f"- Satisfaction: {row[\'satisfaction\']}"
        )
        documents.append(doc)
    return documents, df

def create_vector_store(documents,
                        save_path="data/processed/"):
    print(f"Creating embeddings...")
    print(f"Total documents: {len(documents)}")
    all_embeddings = []
    batch_size = 128
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        result = voyage_client.embed(
            batch,
            model="voyage-3"
        )
        all_embeddings.extend(result.embeddings)
        print(f"Progress: {min(i+batch_size, len(documents))}/{len(documents)}")
    embeddings_array = np.array(all_embeddings).astype("float32")
    dimension = embeddings_array.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_array)
    faiss.write_index(index, f"{save_path}faiss_index.bin")
    with open(f"{save_path}documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"✅ Vector store created!")
    print(f"Total vectors: {index.ntotal:,}")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    return index, documents

def load_vector_store(save_path="data/processed/"):
    index = faiss.read_index(f"{save_path}faiss_index.bin")
    with open(f"{save_path}documents.pkl", "rb") as f:
        documents = pickle.load(f)
    return index, documents

def retrieve_context(query, index, documents, k=5):
    query_embedding = voyage_client.embed(
        [query],
        model="voyage-3"
    ).embeddings[0]
    query_array = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_array, k)
    context = "\\n\\n---\\n\\n".join([
        documents[i] for i in indices[0]
    ])
    return context
'''

with open('src/rag_pipeline.py', 'w') as f:
    f.write(code)
print("✅ rag_pipeline.py written correctly!")

✅ rag_pipeline.py written correctly!


In [20]:
with open('src/rag_pipeline.py', 'r') as f:
    content = f.read()
print(content[:500])

import voyageai
import faiss
import numpy as np
import pandas as pd
import os
import pickle
from dotenv import load_dotenv

load_dotenv()

voyage_client = voyageai.Client(
    api_key=os.getenv("VOYAGE_API_KEY")
)

def prepare_documents(data_path):
    df = pd.read_csv(data_path)
    documents = []
    for _, row in df.iterrows():
        doc = (
            f"Passenger Info:\n"
            f"- Gender: {row['Gender']}\n"
            f"- Age: {row['Age']}\n"
            f"- Customer Type: {row['C


In [21]:
import importlib
import src.rag_pipeline
importlib.reload(src.rag_pipeline)

from src.rag_pipeline import prepare_documents
print("✅ Import successful!")

/Users/nishant/smartflow-rag-chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Import successful!


In [22]:
from src.rag_pipeline import prepare_documents

documents, df = prepare_documents(
    "data/processed/bookings_clean.csv"
)

print(f"Total documents: {len(documents):,}")
print("\nSample document:")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(documents[0])
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

Total documents: 103,594

Sample document:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Passenger Info:
- Gender: Male
- Age: 13
- Customer Type: Loyal Customer
- Type of Travel: Personal Travel
- Class: Eco Plus
- Flight Distance: 460 miles

Service Ratings:
- Wifi: 3/5
- Food: 5/5
- Seat Comfort: 5/5
- Entertainment: 5/5
- On-board Service: 4/5
- Leg Room: 3/5
- Baggage: 4/5
- Checkin: 4/5
- Online Boarding: 3/5

Delays:
- Departure: 25 mins
- Arrival: 18.0 mins

Outcome:
- Satisfaction: neutral or dissatisfied
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [23]:
from src.rag_pipeline import create_vector_store

# Use 1000 docs for now — fast & effective
test_docs = documents[:1000]

print("Starting vector store creation...")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

index, stored_docs = create_vector_store(
    test_docs,
    save_path="data/processed/"
)

Starting vector store creation...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Creating embeddings...
Total documents: 1000


RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

In [24]:
code = '''import voyageai
import faiss
import numpy as np
import pandas as pd
import os
import pickle
import time
from dotenv import load_dotenv

load_dotenv()

voyage_client = voyageai.Client(
    api_key=os.getenv("VOYAGE_API_KEY")
)

def prepare_documents(data_path):
    df = pd.read_csv(data_path)
    documents = []
    for _, row in df.iterrows():
        doc = (
            f"Passenger Info:\\n"
            f"- Gender: {row[\'Gender\']}\\n"
            f"- Age: {row[\'Age\']}\\n"
            f"- Customer Type: {row[\'Customer Type\']}\\n"
            f"- Type of Travel: {row[\'Type of Travel\']}\\n"
            f"- Class: {row[\'Class\']}\\n"
            f"- Flight Distance: {row[\'Flight Distance\']} miles\\n\\n"
            f"Service Ratings:\\n"
            f"- Wifi: {row[\'Inflight wifi service\']}/5\\n"
            f"- Food: {row[\'Food and drink\']}/5\\n"
            f"- Seat Comfort: {row[\'Seat comfort\']}/5\\n"
            f"- Entertainment: {row[\'Inflight entertainment\']}/5\\n"
            f"- On-board Service: {row[\'On-board service\']}/5\\n"
            f"- Leg Room: {row[\'Leg room service\']}/5\\n"
            f"- Baggage: {row[\'Baggage handling\']}/5\\n"
            f"- Checkin: {row[\'Checkin service\']}/5\\n"
            f"- Online Boarding: {row[\'Online boarding\']}/5\\n\\n"
            f"Delays:\\n"
            f"- Departure: {row[\'Departure Delay in Minutes\']} mins\\n"
            f"- Arrival: {row[\'Arrival Delay in Minutes\']} mins\\n\\n"
            f"Outcome:\\n"
            f"- Satisfaction: {row[\'satisfaction\']}"
        )
        documents.append(doc)
    return documents, df

def create_vector_store(documents,
                        save_path="data/processed/"):
    print(f"Creating embeddings...")
    print(f"Total documents: {len(documents)}")
    all_embeddings = []
    batch_size = 20
    
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        try:
            result = voyage_client.embed(
                batch,
                model="voyage-3"
            )
            all_embeddings.extend(result.embeddings)
            print(f"Progress: {min(i+batch_size, len(documents))}/{len(documents)}")
            time.sleep(1)
        except Exception as e:
            print(f"Error at batch {i}: {e}")
            print("Waiting 30 seconds...")
            time.sleep(30)
            result = voyage_client.embed(
                batch,
                model="voyage-3"
            )
            all_embeddings.extend(result.embeddings)
    
    embeddings_array = np.array(all_embeddings).astype("float32")
    dimension = embeddings_array.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_array)
    faiss.write_index(index, f"{save_path}faiss_index.bin")
    with open(f"{save_path}documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"✅ Vector store created!")
    print(f"Total vectors: {index.ntotal:,}")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    return index, documents

def load_vector_store(save_path="data/processed/"):
    index = faiss.read_index(f"{save_path}faiss_index.bin")
    with open(f"{save_path}documents.pkl", "rb") as f:
        documents = pickle.load(f)
    return index, documents

def retrieve_context(query, index, documents, k=5):
    query_embedding = voyage_client.embed(
        [query],
        model="voyage-3"
    ).embeddings[0]
    query_array = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_array, k)
    context = "\\n\\n---\\n\\n".join([
        documents[i] for i in indices[0]
    ])
    return context
'''

with open('src/rag_pipeline.py', 'w') as f:
    f.write(code)
print("✅ rag_pipeline.py updated!")

✅ rag_pipeline.py updated!


In [26]:
import importlib
import src.rag_pipeline
importlib.reload(src.rag_pipeline)

from src.rag_pipeline import (
    prepare_documents,
    create_vector_store
)

# Reload documents
documents, df = prepare_documents(
    "data/processed/bookings_clean.csv"
)
print(f"✅ {len(documents):,} documents ready")

✅ 103,594 documents ready


In [27]:
# Use 1000 docs — good balance
# of speed and quality
test_docs = documents[:1000]

index, stored_docs = create_vector_store(
    test_docs,
    save_path="data/processed/"
)

Creating embeddings...
Total documents: 1000
Progress: 20/1000
Progress: 40/1000
Progress: 60/1000
Progress: 80/1000
Progress: 100/1000
Progress: 120/1000
Progress: 140/1000
Progress: 160/1000
Progress: 180/1000
Progress: 200/1000
Progress: 220/1000
Progress: 240/1000
Progress: 260/1000
Progress: 280/1000
Progress: 300/1000
Progress: 320/1000
Progress: 340/1000
Progress: 360/1000
Progress: 380/1000
Progress: 400/1000
Progress: 420/1000
Progress: 440/1000
Progress: 460/1000
Progress: 480/1000
Progress: 500/1000
Progress: 520/1000
Progress: 540/1000
Progress: 560/1000
Progress: 580/1000
Progress: 600/1000
Progress: 620/1000
Progress: 640/1000
Progress: 660/1000
Progress: 680/1000
Progress: 700/1000
Progress: 720/1000
Progress: 740/1000
Progress: 760/1000
Progress: 780/1000
Progress: 800/1000
Progress: 820/1000
Progress: 840/1000
Progress: 860/1000
Progress: 880/1000
Progress: 900/1000
Progress: 920/1000
Progress: 940/1000
Progress: 960/1000
Progress: 980/1000
Progress: 1000/1000
━━━━━━━━

In [28]:
from src.rag_pipeline import (
    load_vector_store,
    retrieve_context
)

# Load saved vector store
index, stored_docs = load_vector_store(
    save_path="data/processed/"
)

# Test retrieval
query = "How satisfied are business class passengers?"
context = retrieve_context(
    query, index, stored_docs
)

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Query: {query}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("Retrieved Context:")
print(context[:500])
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Query: How satisfied are business class passengers?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Retrieved Context:
Passenger Info:
- Gender: Female
- Age: 24
- Customer Type: Loyal Customer
- Type of Travel: Business travel
- Class: Business
- Flight Distance: 1013 miles

Service Ratings:
- Wifi: 3/5
- Food: 3/5
- Seat Comfort: 3/5
- Entertainment: 3/5
- On-board Service: 1/5
- Leg Room: 1/5
- Baggage: 2/5
- Checkin: 2/5
- Online Boarding: 3/5

Delays:
- Departure: 25 mins
- Arrival: 13.0 mins

Outcome:
- Satisfaction: satisfied

---

Passenger Info:
- Gender: Female
- Age: 42
- Customer Type: Loyal Customer
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
